# Gauss Initial Orbit Determination (IOD)

This tutorial demonstrates **Gauss's Method for Initial Orbit Determination**, which allows full state estimation (position and velocity) from three line-of-sight (LOS) observations from a ground station or another spacecraft.

## Theory Background

Given three unit vectors $\hat{\rho}_1, \hat{\rho}_2, \hat{\rho}_3$ at times $t_1, t_2, t_3$ and the observer position vectors $R_1, R_2, R_3$, we find the spacecraft position vectors:

$$r_i = R_i + \rho_i \hat{\rho}_i \quad (i=1,2,3)$$

Gauss's method exploits the fact that the three position vectors lie in the orbital plane, so one is a linear combination of the other two:

$$r_2 = c_1 r_1 + c_3 r_3$$

Where the coefficients $c_1$ and $c_3$ are approximated using time intervals $\tau_k = t_k - t_2$ and the magnitude of the position vector $r_2$:

$$c_1 \approx \frac{\tau_3}{\tau_3 - \tau_1} \left(1 + \frac{\mu}{6 r_2^3} (\tau_3^2 - \tau_1^2)\right)$$
$$c_3 \approx \frac{-\tau_1}{\tau_3 - \tau_1} \left(1 + \frac{\mu}{6 r_2^3} (\tau_3^2 - \tau_1^2)\right)$$

Substituting these into the Coplanar Equation leads to a scalar equation for the distance $\rho_2$, which yields the full state.

In [ ]:
import numpy as np
from gnc_toolkit.navigation.iod import gauss_iod
from gnc_toolkit.utils.state_to_elements import kepler2eci, eci2kepler

print("--- Gauss Initial Orbit Determination (IOD) ---/n")

# 1. Define a ground truth orbit (ISS-like LEO)
mu = 398600.4415e9
a = 6738e3       # Semi-major axis (m)
ecc = 0.001      # Eccentricity
incl = np.radians(51.64)
raan = np.radians(45.0)
argp = np.radians(30.0)
nu0 = np.radians(0.0)

# 2. Generate observations at three time steps
dt = 300.0  # 5 minutes arc
t1, t2, t3 = 0.0, dt, 2.0 * dt

mean_motion = np.sqrt(mu / a**3)
nu1 = nu0
nu2 = nu0 + mean_motion * t2
nu3 = nu0 + mean_motion * t3

r1_true, v1_true = kepler2eci(a, ecc, incl, raan, argp, nu1)
r2_true, v2_true = kepler2eci(a, ecc, incl, raan, argp, nu2)
r3_true, v3_true = kepler2eci(a, ecc, incl, raan, argp, nu3)

# 3. Observer positions (sitting at Equator)
Re = 6378e3
R1 = np.array([Re, 0.0, 0.0])
R2 = np.array([Re, 0.0, 0.0])
R3 = np.array([Re, 0.0, 0.0])

# 4. Line-of-Sight (LOS) vectors
rho1_vec = r1_true - R1
rho2_vec = r2_true - R2
rho3_vec = r3_true - R3

rho_hat1 = rho1_vec / np.linalg.norm(rho1_vec)
rho_hat2 = rho2_vec / np.linalg.norm(rho2_vec)
rho_hat3 = rho3_vec / np.linalg.norm(rho3_vec)

# 5. Perform Gauss IOD
state_est = gauss_iod(rho_hat1, rho_hat2, rho_hat3, t1, t2, t3, R1, R2, R3, mu=mu)
r2_est = state_est[:3]
v2_est = state_est[3:]

# 6. Results comparison
print("Results at Epoch t2:")
print(f"Position Error: {np.linalg.norm(r2_est - r2_true):.2f} m")
print(f"Velocity Error: {np.linalg.norm(v2_est - v2_true):.4f} m/s")

a_est, ecc_est = eci2kepler(r2_est, v2_est)[:2]
print(f"Est Semi-major axis: {a_est/1e3:.2f} km (True: {a/1e3:.2f} km)")
print(f"Est Eccentricity: {ecc_est:.6f} (True: {ecc:.6f})")